# Phase 2.3 — Exploratory Data Analysis: Training Metadata (`2014–2025`)

**Project:** G2F Maize Yield Prediction 2026  
**Primary Dataset:** `2_Training_Meta_Data_2014_2025.csv`  
**Cross-Reference (Read-Only Keys Only):** `1_Training_Trait_Data_2014_2025.csv` (`Env`, `Year`, `Field_Location`, `Yield_Mg_ha`)

## What Is Metadata and Why Does It Matter for Machine Learning?
In multi-environment agricultural trials, **metadata** describes *where*, *when*, and *how* each field trial was conducted — such as GPS coordinates, weather-station placement, irrigation, tillage, previous crop, planter type, and field-level comments.

In a Genotype × Environment (G × E) machine learning pipeline, environment metadata can:
1. Provide **join keys** (`Env`) connecting plot-level yields to environmental context (weather, soil, environmental covariates).
2. Supply **spatial and management context** (e.g., latitude/longitude, irrigation, treatment, crop rotation) — *only if* verified to be available at 2026 prediction time.
3. Expose **data quality and leakage risks** (e.g., post-harvest equipment notes, in-season issue logs, inconsistent naming, or missing environments).

## Scope & Guardrails
- Strictly read-only EDA: no raw files are modified, cleaned, imputed, or merged into persisted datasets.
- No production feature engineering or model training is performed.
- All numerical statistics and chart titles are computed dynamically at runtime.
- Per project data-sharing policy (`DEC-007`), committed versions of this notebook have all outputs stripped; local execution saves visualizations under `reports/figures/local/`.

## 1. Reproducible Data Loading

We resolve all paths relative to the repository root so the notebook runs identically on any machine. Following our defensive CSV protocol, we inspect the header and first 5 rows (`nrows=5`) with Pandas' C engine before loading the complete file, then record exact load time and deep memory usage while preserving all original column names and values.

In [ ]:
import math
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root(start: Path) -> Path:
    """Walk upward from start until the repository root (containing README.md and data/) is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing README.md and data/.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_TRAIN_DIR = PROJECT_ROOT / "data/raw/training/Training_Data-20261123T123657Z-1-001/Training_Data"
META_PATH = RAW_TRAIN_DIR / "2_Training_Meta_Data_2014_2025.csv"
TRAIT_PATH = RAW_TRAIN_DIR / "1_Training_Trait_Data_2014_2025.csv"
LOCAL_FIGURES_DIR = PROJECT_ROOT / "reports/figures/local"
LOCAL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert META_PATH.is_file(), f"Metadata file not found: {META_PATH.relative_to(PROJECT_ROOT)}"
assert TRAIT_PATH.is_file(), f"Trait file not found: {TRAIT_PATH.relative_to(PROJECT_ROOT)}"

print(f"Metadata relative path : {META_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Metadata file size     : {META_PATH.stat().st_size / (1024 ** 2):.4f} MB ({META_PATH.stat().st_size:,} bytes)")
print(f"Local figures folder   : {LOCAL_FIGURES_DIR.relative_to(PROJECT_ROOT).as_posix()}")

In [ ]:
# Step 1a: Inspect header and first 5 sample rows before full load
preview_meta = pd.read_csv(META_PATH, engine="c", nrows=5)
print(f"Preview shape: {preview_meta.shape}")
print(f"Column count : {len(preview_meta.columns)}")
display(preview_meta)

In [ ]:
# Step 1b: Full load using Pandas C engine with low_memory=False (preserving original column names and values)
t0 = time.perf_counter()
meta_df = pd.read_csv(META_PATH, engine="c", low_memory=False)
load_seconds = time.perf_counter() - t0

mem_bytes_deep = int(meta_df.memory_usage(deep=True).sum())
mem_mb_deep = mem_bytes_deep / (1024 ** 2)
mem_kb_deep = mem_bytes_deep / 1024

print(f"Loaded shape          : {meta_df.shape[0]:,} rows x {meta_df.shape[1]} columns")
print(f"Load time (C engine)  : {load_seconds:.4f} seconds")
print(f"Memory usage (deep)   : {mem_mb_deep:.4f} MB ({mem_kb_deep:.2f} KB | {mem_bytes_deep:,} bytes)")

## 2. Dataset Structure, Missingness, Sentinels & Column Meanings

### Why this matters for machine learning
Metadata tables are small in row count but wide and often filled by human collaborators at dozens of field sites. Auditing missingness, exact duplicates, and sentinel tokens (`"NA"`, `"unknown"`, `"None"`, `"-999"`, etc.) early prevents silent parsing bugs and reveals which fields are too sparse or free-text to use naively.

In [ ]:
# Full structure, cardinality, and missingness audit across all columns
structure_df = pd.DataFrame({
    "column": meta_df.columns,
    "inferred_dtype": meta_df.dtypes.astype(str).values,
    "non_missing": meta_df.notna().sum().values,
    "missing_count": meta_df.isna().sum().values,
    "missing_pct": (meta_df.isna().mean() * 100).round(2).values,
    "nunique_non_null": [int(meta_df[c].nunique(dropna=True)) for c in meta_df.columns],
    "memory_kb_deep": [round(meta_df[c].memory_usage(deep=True) / 1024, 2) for c in meta_df.columns],
})

exact_dups = int(meta_df.duplicated().sum())
print(f"Total rows: {len(meta_df):,} | Total columns: {meta_df.shape[1]} | Exact duplicate rows: {exact_dups:,}")
print("\n=== COMPLETE METADATA SCHEMA & MISSINGNESS ===")
print(structure_df.to_string(index=False))

In [ ]:
# Audit potential missing-value sentinel strings and numerical codes WITHOUT replacing any values
candidate_str_sentinels = {
    "", " ", "na", "n/a", "nan", "null", "none", ".", "-", "--", "?", "missing",
    "unknown", "unk", "nd", "n.d.", "not available", "not applicable", "not known",
    "tbd", "-999", "-99", "999", "9999",
}

print("=== 1) CANDIDATE SENTINEL STRINGS IN PARSED NON-NULL STRING VALUES ===")
parsed_hits = {}
for col in meta_df.select_dtypes(include=["object", "string"]).columns:
    s = meta_df[col].dropna().astype(str).str.strip()
    mask = s.str.lower().isin(candidate_str_sentinels)
    if mask.any():
        parsed_hits[col] = s[mask].value_counts().to_dict()
if parsed_hits:
    for col, counts in parsed_hits.items():
        print(f"  {col}: {counts}")
else:
    print("  None detected in non-null parsed string values.")

print("\n=== 2) RAW NA-LIKE TOKENS IN CSV (keep_default_na=False, read-only) ===")
raw_str_df = pd.read_csv(META_PATH, engine="c", dtype=str, keep_default_na=False, low_memory=False)
pandas_default_na = {
    "", "#N/A", "#N/A N/A", "#NA", "-1.#IND", "-1.#QNAN", "-NaN", "-nan",
    "1.#IND", "1.#QNAN", "<NA>", "N/A", "NA", "NULL", "NaN", "None", "n/a", "nan", "null"
}
for col in raw_str_df.columns:
    s = raw_str_df[col]
    hits = s[s.isin(pandas_default_na) | s.str.strip().str.lower().isin(candidate_str_sentinels)]
    if len(hits) > 0:
        vc = hits.value_counts().to_dict()
        # Only print non-empty tokens or summarize empty-string count cleanly
        non_empty = {k: v for k, v in vc.items() if k != ""}
        empty_n = vc.get("", 0)
        if non_empty:
            print(f"  {col}: empty='' ({empty_n}), other_na_tokens={non_empty}")
del raw_str_df

print("\n=== 3) NUMERICAL FINITE & SENTINEL CODE AUDIT ===")
num_codes = [-9999, -999, -99, -1, 0, 999, 9999]
num_rows = []
for col in meta_df.select_dtypes(include=[np.number]).columns:
    raw_num = pd.to_numeric(meta_df[col], errors="coerce")
    n_pos_inf = int(np.isposinf(raw_num).sum())
    n_neg_inf = int(np.isneginf(raw_num).sum())
    finite = raw_num[np.isfinite(raw_num)]
    hits = {c: int((finite == c).sum()) for c in num_codes if (finite == c).any()}
    num_rows.append({
        "column": col,
        "n_finite": int(len(finite)),
        "pos_inf": n_pos_inf,
        "neg_inf": n_neg_inf,
        "min": round(float(finite.min()), 6) if len(finite) else np.nan,
        "max": round(float(finite.max()), 6) if len(finite) else np.nan,
        "code_hits": str(hits) if hits else "None",
    })
print(pd.DataFrame(num_rows).to_string(index=False))

In [ ]:
# Local aggregate missingness visualization
plot_miss = structure_df[structure_df["missing_count"] > 0].sort_values("missing_pct", ascending=True)

fig, ax = plt.subplots(figsize=(11, 10))
bars = ax.barh(plot_miss["column"], plot_miss["missing_pct"], color="#2b6cb0", edgecolor="white")
ax.set_xlabel("Missing Values (%)")
ax.set_title(f"Missing Value Percentage by Column — {META_PATH.name} (n = {len(meta_df):,})")
ax.set_xlim(0, 118)

for bar, (_, row) in zip(bars, plot_miss.iterrows()):
    ax.text(
        bar.get_width() + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{row['missing_pct']:.1f}% ({int(row['missing_count'])})",
        va="center",
        fontsize=8,
    )

plt.tight_layout()
fig_path_1 = LOCAL_FIGURES_DIR / "01_meta_missing_values.png"
fig.savefig(fig_path_1, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved local figure: {fig_path_1.relative_to(PROJECT_ROOT).as_posix()}")

### Scientific Meaning of Metadata Columns (Verified vs. Inferred)

> **Documentation Boundary:** No separate data dictionary file is present inside `data/raw/training/`. Official EvalAI competition documentation (`https://eval.ai/api/challenges/challenge/2742/`) confirms that metadata/field information covers **field GPS locations and field information**, and several CSV headers themselves include explicit units or parenthetical definitions (marked **Verified from header**). All other semantic interpretations below are **Inferred** from G2F/agronomic conventions and should be confirmed against official G2F documentation.

| Column | Documentation Status | Meaning / Interpretation |
| :--- | :--- | :--- |
| `Year` | Inferred | Growing season calendar year of the trial. |
| `Env` | Inferred | Unique field trial environment identifier (`<Location/Sub-trial>_<Year>`), primary join key across G2F tables. |
| `Experiment_Code` | Inferred | Site/experiment code associated with the cooperator location or trial series. |
| `Treatment` | Inferred | Experimental agronomic treatment (e.g., standard, drought, dryland, irrigated, early/late planting, low N, disease trial). |
| `City` | Inferred | Nearest city/town of the field trial location. |
| `Farm` | Inferred | Research station or farm name where the trial was planted. |
| `Field` | Inferred | Specific field/parcel identifier within the farm. |
| `Trial_ID (Assigned by collaborator for internal reference)` | Verified from header (purpose); details inferred | Internal trial identifier assigned by the local field collaborator. |
| `Soil_Taxonomic_ID and horizon description, if known` | Verified from header (scope) | Free-text soil taxonomic classification and soil horizon notes reported by collaborators. |
| `Weather_Station_Serial_Number (Last four digits, e.g. m2700s#####)` | Verified from header (format) | Hardware serial identifier (last 4 digits) of the on-site weather station. |
| `Weather_Station_Latitude (in decimal numbers NOT DMS)` | Verified from EvalAI + header | Decimal-degree latitude of the trial weather station (WGS84 assumed). |
| `Weather_Station_Longitude (in decimal numbers NOT DMS)` | Verified from EvalAI + header | Decimal-degree longitude of the trial weather station (WGS84 assumed). |
| `Date_weather_station_placed` | Inferred | Date the on-site weather station was deployed in the field. |
| `Date_weather_station_removed` | Inferred | Date the on-site weather station was removed from the field (typically post-harvest). |
| `Previous_Crop` | Inferred | Crop grown in the same field during the preceding season (rotation context). |
| `Pre-plant_tillage_method(s)` | Inferred | Tillage operations performed before planting. |
| `In-season_tillage_method(s)` | Inferred | Cultivation/tillage operations performed during the growing season. |
| `Type_of_planter (fluted cone; belt cone; air planter)` | Verified from header (examples) | Planter equipment type used to sow plots. |
| `System_Determining_Moisture` | Inferred | Grain moisture measurement instrument/combine sensor system used at harvest. |
| `Pounds_Needed_Grain_Moisture` | Inferred | Minimum harvested grain sample weight (lbs) required for reliable moisture sensor reading. |
| `Latitude_of_Field_Corner_#1..#4` (4 columns) | Verified from EvalAI + header | Decimal-degree latitudes of the 4 bounding corners of the trial field (lower left, lower right, upper right, upper left). |
| `Longitude_of_Field_Corner_#1..#4` (4 columns) | Verified from EvalAI + header | Decimal-degree longitudes of the 4 bounding corners of the trial field. |
| `Cardinal_Heading_Pass_1` | Inferred | Compass direction (e.g., N, S, E, W, degrees) along Pass 1 of the field plot grid. |
| `Irrigated` | Inferred | Binary indicator (`yes` / `no`) of whether the trial received irrigation. |
| `Issue/comment_#1` – `Issue/comment_#8` (8 columns) | Inferred | Free-text collaborator notes recording field events, weather extremes, management notes, or trial anomalies. |

## 3. Environment Identifiers & Observation Granularity

### Why dataset keys matter in machine learning
A **primary key** uniquely identifies every row in a reference table. Before joining metadata to plot-level trait records in future phases, we must prove whether `Env` is strictly unique in metadata (1 row per `Env`), or whether duplicate `Env` rows exist that would silently multiply trait rows during a join!

In [ ]:
# Check Env uniqueness and identifier cardinality
n_rows = len(meta_df)
n_unique_env = int(meta_df["Env"].nunique(dropna=True))
env_is_unique = bool(meta_df["Env"].is_unique)
dup_env_rows = int(meta_df.duplicated(subset=["Env"], keep=False).sum())

print("=== ENVIRONMENT GRANULARITY AUDIT ===")
print(f"Total metadata rows              : {n_rows:,}")
print(f"Unique Env identifiers           : {n_unique_env:,}")
print(f"Env uniquely identifies each row : {env_is_unique}")
print(f"Rows with duplicate Env          : {dup_env_rows:,}")
print(f"Unique Year values               : {meta_df['Year'].nunique()} ({int(meta_df['Year'].min())}–{int(meta_df['Year'].max())})")
print(f"Unique Experiment_Code values    : {meta_df['Experiment_Code'].nunique():,}")
print(f"Unique City values (raw)         : {meta_df['City'].nunique(dropna=True):,}")
print(f"Unique City values (stripped/low): {meta_df['City'].dropna().astype(str).str.strip().str.lower().nunique():,}")

# Environment coverage by year
env_by_year = (
    meta_df.groupby("Year")
    .agg(
        n_envs=("Env", "nunique"),
        n_exp_codes=("Experiment_Code", "nunique"),
        n_cities=("City", "nunique"),
        has_station_coords=("Weather_Station_Latitude (in decimal numbers NOT DMS)", "count"),
        has_corner1_coords=("Latitude_of_Field_Corner_#1 (lower left)", "count"),
    )
)
print("\n=== METADATA ENVIRONMENT COVERAGE BY YEAR ===")
print(env_by_year.to_string())

expected_years = set(range(2014, 2026))
observed_years = set(meta_df["Year"].unique())
print(f"\nAll years 2014-2025 covered: {expected_years == observed_years}")

In [ ]:
# Audit potential inconsistencies in Env, Year, and Experiment_Code WITHOUT modifying identifiers
env_year_suffix = meta_df["Env"].str.extract(r"_(\d{4})$")[0]
env_suffix_missing = int(env_year_suffix.isna().sum())
env_year_mismatch = int((env_year_suffix.astype("Int64") != meta_df["Year"]).sum())

env_prefix = meta_df["Env"].str.replace(r"_\d{4}$", "", regex=True)
prefix_vs_exp_mismatch = meta_df[env_prefix != meta_df["Experiment_Code"]]

# Check multiple Envs sharing the same (Year, Experiment_Code)
multi_env_per_exp_yr = (
    meta_df.groupby(["Year", "Experiment_Code"])["Env"]
    .agg(n_envs="count", envs=lambda s: ", ".join(sorted(s)))
    .query("n_envs > 1")
)

# Check whitespace / hyphenation inconsistencies in Experiment_Code
exp_codes = sorted(meta_df["Experiment_Code"].dropna().unique())
exp_norm = pd.Series(exp_codes).str.replace(r"\s+", "", regex=True).str.lower()
exp_norm_groups = pd.DataFrame({"raw": exp_codes, "norm": exp_norm}).groupby("norm")["raw"].apply(list)
exp_inconsistent_variants = exp_norm_groups[exp_norm_groups.apply(len) > 1]

print("=== IDENTIFIER CONSISTENCY AUDIT ===")
print(f"Env values missing _YYYY suffix               : {env_suffix_missing}")
print(f"Env _YYYY suffix != Year column               : {env_year_mismatch}")
print(f"Env prefix != Experiment_Code rows            : {len(prefix_vs_exp_mismatch)}")
if len(prefix_vs_exp_mismatch) > 0:
    print(prefix_vs_exp_mismatch[["Year", "Env", "Experiment_Code", "Treatment"]].to_string(index=False))

print(f"\n(Year, Experiment_Code) groups with >1 Env    : {len(multi_env_per_exp_yr)}")
if len(multi_env_per_exp_yr) > 0:
    print(multi_env_per_exp_yr.to_string())

print(f"\nExperiment_Code spacing/casing variant groups : {len(exp_inconsistent_variants)}")
if len(exp_inconsistent_variants) > 0:
    print(exp_inconsistent_variants.to_string())

In [ ]:
# Local visualization: Environment coverage by year
fig, ax = plt.subplots(figsize=(10, 4.5))
yrs = env_by_year.index.values
ax.bar(yrs, env_by_year["n_envs"], color="#2f855a", alpha=0.85, label="Total Envs in Metadata")
ax.plot(yrs, env_by_year["has_station_coords"], "-o", color="#1a365d", linewidth=1.8, label="Envs with Weather-Station GPS")
ax.plot(yrs, env_by_year["has_corner1_coords"], "--s", color="#dd6b20", linewidth=1.6, label="Envs with Field Corner #1 GPS")
ax.set_title(f"Metadata Environment & Coordinate Coverage by Year ({int(yrs.min())}–{int(yrs.max())})")
ax.set_xlabel("Growing Season Year")
ax.set_ylabel("Number of Environments")
ax.set_xticks(yrs)
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=True)

plt.tight_layout()
fig_path_2 = LOCAL_FIGURES_DIR / "02_meta_env_coverage_by_year.png"
fig.savefig(fig_path_2, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved local figure: {fig_path_2.relative_to(PROJECT_ROOT).as_posix()}")

## 4. Geographical Analysis

### Why geographical features matter for machine learning
Latitude and longitude encode macro-climate gradients (day length / photoperiod, heat units, frost-free season length, continentality). However, raw coordinates also pose two risks:
1. **Coordinate errors:** Missing values, sign flips (missing minus sign in the Western Hemisphere), swapped latitude/longitude, or out-of-range degrees.
2. **Spatial overfitting vs. generalization:** A tree model can memorize exact coordinates of historical sites; checking cross-year coordinate drift and location naming consistency helps us plan spatially robust features later.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized Haversine great-circle distance in kilometers."""
    r = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


ws_lat_col = "Weather_Station_Latitude (in decimal numbers NOT DMS)"
ws_lon_col = "Weather_Station_Longitude (in decimal numbers NOT DMS)"
corner_lat_cols = [c for c in meta_df.columns if c.startswith("Latitude_of_Field_Corner")]
corner_lon_cols = [c for c in meta_df.columns if c.startswith("Longitude_of_Field_Corner")]
all_lat_cols = [ws_lat_col, *corner_lat_cols]
all_lon_cols = [ws_lon_col, *corner_lon_cols]

# Coverage summary
has_ws = meta_df[ws_lat_col].notna() & meta_df[ws_lon_col].notna()
has_any_corner = meta_df[corner_lat_cols].notna().any(axis=1) & meta_df[corner_lon_cols].notna().any(axis=1)
has_all_corners = meta_df[corner_lat_cols].notna().all(axis=1) & meta_df[corner_lon_cols].notna().all(axis=1)
has_any_geo = has_ws | has_any_corner

print("=== GEOGRAPHICAL COORDINATE COVERAGE ===")
print(f"Total environments                     : {len(meta_df):,}")
print(f"Complete Weather Station (lat & lon)   : {int(has_ws.sum()):,} ({has_ws.mean() * 100:.2f}%)")
print(f"At least 1 Field Corner (lat & lon)    : {int(has_any_corner.sum()):,} ({has_any_corner.mean() * 100:.2f}%)")
print(f"All 4 Field Corners complete           : {int(has_all_corners.sum()):,} ({has_all_corners.mean() * 100:.2f}%)")
print(f"Any valid coordinate (WS or Corner)    : {int(has_any_geo.sum()):,} ({has_any_geo.mean() * 100:.2f}%)")
print(f"No coordinates at all                  : {int((~has_any_geo).sum()):,} ({(~has_any_geo).mean() * 100:.2f}%)")

# Range and validity audit
geo_rows = []
for col in all_lat_cols + all_lon_cols:
    s = pd.to_numeric(meta_df[col], errors="coerce")
    fin = s[np.isfinite(s)]
    is_lat = "Latitude" in col
    invalid_world = int(((fin < -90) | (fin > 90)).sum()) if is_lat else int(((fin < -180) | (fin > 180)).sum())
    n_zero = int((fin == 0).sum())
    n_pos_lon = int((fin > 0).sum()) if not is_lat else 0
    geo_rows.append({
        "column": col,
        "non_null": int(len(fin)),
        "missing": int(s.isna().sum()),
        "min": round(float(fin.min()), 6),
        "max": round(float(fin.max()), 6),
        "invalid_world_bounds": invalid_world,
        "zero_values": n_zero,
        "positive_lon_count": n_pos_lon,
    })
print("\n=== COORDINATE RANGE & BOUNDS AUDIT ===")
print(pd.DataFrame(geo_rows).to_string(index=False))

# Inspect positive longitude rows (verify whether legitimate European sites vs. sign errors)
pos_lon_mask = meta_df[all_lon_cols].gt(0).any(axis=1)
print(f"\nEnvironments with positive longitude (Eastern Hemisphere): {int(pos_lon_mask.sum())}")
if pos_lon_mask.any():
    print(meta_df.loc[pos_lon_mask, ["Year", "Env", "Experiment_Code", "City", ws_lat_col, ws_lon_col]].to_string(index=False))

# Check corner lat/lon pairing mismatches (e.g., lat missing while lon present or vice versa)
print("\n=== LAT/LON PAIRING ASYMMETRY CHECK ===")
for lat_c, lon_c in zip(all_lat_cols, all_lon_cols):
    asym = int((meta_df[lat_c].notna() != meta_df[lon_c].notna()).sum())
    if asym > 0:
        print(f"  Asymmetric missingness ({asym} rows): {lat_c} vs {lon_c}")
        print(meta_df.loc[meta_df[lat_c].notna() != meta_df[lon_c].notna(), ["Year", "Env", lat_c, lon_c]].to_string(index=False))

In [ ]:
# Diagnostic: Weather Station vs. Field Corner Centroid distance + Cross-year coordinate consistency per location
corner_lat_mean = meta_df[corner_lat_cols].mean(axis=1)
corner_lon_mean = meta_df[corner_lon_cols].mean(axis=1)

both_ws_and_corner = has_ws & corner_lat_mean.notna() & corner_lon_mean.notna()
ws_to_field_km = haversine_km(
    meta_df.loc[both_ws_and_corner, ws_lat_col],
    meta_df.loc[both_ws_and_corner, ws_lon_col],
    corner_lat_mean.loc[both_ws_and_corner],
    corner_lon_mean.loc[both_ws_and_corner],
)
print("=== WEATHER STATION TO FIELD CENTROID DISTANCE (km, diagnostic only) ===")
print(f"Environments with both WS and Field Corner coords: {int(both_ws_and_corner.sum())}")
print(ws_to_field_km.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(4).to_string())
far_ws = meta_df.loc[both_ws_and_corner, ["Year", "Env", "City"]].assign(dist_km=ws_to_field_km.round(3))
print("\nTop 10 largest Weather-Station-to-Field distances (km):")
print(far_ws.sort_values("dist_km", ascending=False).head(10).to_string(index=False))

# Cross-year coordinate consistency by Experiment_Code (using best available coordinate: WS else Corner centroid)
best_lat = meta_df[ws_lat_col].fillna(corner_lat_mean)
best_lon = meta_df[ws_lon_col].fillna(corner_lon_mean)
geo_check = meta_df[["Year", "Env", "Experiment_Code", "City"]].assign(lat=best_lat, lon=best_lon).dropna(subset=["lat", "lon"])

loc_spread_rows = []
for exp_code, grp in geo_check.groupby("Experiment_Code"):
    if len(grp) < 2:
        continue
    lat_span = float(grp["lat"].max() - grp["lat"].min())
    lon_span = float(grp["lon"].max() - grp["lon"].min())
    # Max distance from location median coordinate (km)
    med_lat, med_lon = float(grp["lat"].median()), float(grp["lon"].median())
    max_dev_km = float(haversine_km(grp["lat"], grp["lon"], med_lat, med_lon).max())
    loc_spread_rows.append({
        "Experiment_Code": exp_code,
        "n_envs_with_gps": len(grp),
        "n_years": int(grp["Year"].nunique()),
        "n_cities_raw": int(grp["City"].nunique(dropna=True)),
        "lat_span_deg": round(lat_span, 5),
        "lon_span_deg": round(lon_span, 5),
        "max_dev_from_median_km": round(max_dev_km, 3),
    })

loc_spread_df = pd.DataFrame(loc_spread_rows).sort_values("max_dev_from_median_km", ascending=False)
print(f"\n=== CROSS-YEAR COORDINATE CONSISTENCY BY Experiment_Code (n={len(loc_spread_df)} multi-env codes) ===")
print(f"Codes with max deviation > 10 km from median : {int((loc_spread_df['max_dev_from_median_km'] > 10).sum())}")
print(f"Codes with max deviation > 50 km from median : {int((loc_spread_df['max_dev_from_median_km'] > 50).sum())}")
print("\nTop 12 Experiment_Codes by cross-year coordinate spread:")
print(loc_spread_df.head(12).to_string(index=False))

# Location naming inconsistency audit (City per Experiment_Code and City capitalization/punctuation variants)
city_clean = meta_df["City"].dropna().astype(str).str.strip()
city_norm = city_clean.str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
city_variants = pd.DataFrame({"raw": city_clean, "norm": city_norm}).drop_duplicates().groupby("norm")["raw"].apply(list)
city_variants = city_variants[city_variants.apply(len) > 1]

exp_multi_city = meta_df.dropna(subset=["City"]).groupby("Experiment_Code")["City"].unique()
exp_multi_city = exp_multi_city[exp_multi_city.apply(len) > 1]

print(f"\n=== LOCATION NAMING INCONSISTENCIES ===")
print(f"City spelling/casing variant groups             : {len(city_variants)}")
if len(city_variants) > 0:
    print(city_variants.to_string())
print(f"\nExperiment_Codes linked to multiple City values : {len(exp_multi_city)}")
if len(exp_multi_city) > 0:
    for k, vals in exp_multi_city.items():
        print(f"  {k}: {list(vals)}")

In [ ]:
# Local geographical scatter plots (Matplotlib only — no external mapping libraries or paid APIs)
na_geo = geo_check[geo_check["lon"] < 0]
eu_geo = geo_check[geo_check["lon"] >= 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.8, 1]})

sc = axes[0].scatter(
    na_geo["lon"],
    na_geo["lat"],
    c=na_geo["Year"],
    cmap="viridis",
    alpha=0.75,
    s=32,
    edgecolors="black",
    linewidths=0.3,
)
axes[0].set_title(f"North American Trial Coordinates (n = {len(na_geo)} envs)")
axes[0].set_xlabel("Longitude (deg)")
axes[0].set_ylabel("Latitude (deg)")
cbar = plt.colorbar(sc, ax=axes[0])
cbar.set_label("Year")

axes[1].scatter(geo_check["lon"], geo_check["lat"], color="#2b6cb0", alpha=0.7, s=28, label=f"Western Hem. (n={len(na_geo)})")
if len(eu_geo) > 0:
    axes[1].scatter(eu_geo["lon"], eu_geo["lat"], color="#c53030", marker="^", s=55, label=f"Eastern Hem. (n={len(eu_geo)})")
axes[1].set_title(f"Global View of All GPS Coordinates (n = {len(geo_check)} envs)")
axes[1].set_xlabel("Longitude (deg)")
axes[1].set_ylabel("Latitude (deg)")
axes[1].legend(frameon=True, fontsize=8.5)

plt.tight_layout()
fig_path_3 = LOCAL_FIGURES_DIR / "03_meta_geo_coordinates.png"
fig.savefig(fig_path_3, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved local figure: {fig_path_3.relative_to(PROJECT_ROOT).as_posix()}")

## 5. Agricultural, Date & Management Information

### Schema Verification Note
First we explicitly verify whether planting dates, harvest dates, or plot-area fields exist in `2_Training_Meta_Data_2014_2025.csv`:
- **Planting date, harvest date, and plot-area columns are NOT present** in `2_Training_Meta_Data_2014_2025.csv` (those fields reside in `1_Training_Trait_Data_2014_2025.csv`, analyzed in Phase 2.2).
- The two date columns present in metadata are **`Date_weather_station_placed`** and **`Date_weather_station_removed`**, which we audit for formats, parsing validity, year consistency, and station-deployment duration (strictly as a diagnostic calculation — not persisted).
- We then audit all available agronomic and field-management fields (`Treatment`, `Irrigated`, `Previous_Crop`, `Pre-plant_tillage_method(s)`, `In-season_tillage_method(s)`, `Type_of_planter`, `Cardinal_Heading_Pass_1`, `Soil_Taxonomic_ID`, moisture-measurement equipment, and `Issue/comment_#1..#8`).

In [ ]:
# Explicitly verify presence/absence of planting date, harvest date, and plot-area columns in metadata
planting_cols = [c for c in meta_df.columns if "plant" in c.lower() and "date" in c.lower()]
harvest_cols = [c for c in meta_df.columns if "harvest" in c.lower()]
plot_area_cols = [c for c in meta_df.columns if "area" in c.lower() or "plot" in c.lower()]
date_cols = [c for c in meta_df.columns if "date" in c.lower()]

print("=== DATE & PLOT-AREA COLUMN AVAILABILITY IN METADATA ===")
print(f"Planting date columns : {planting_cols}")
print(f"Harvest date columns  : {harvest_cols}")
print(f"Plot-area columns     : {plot_area_cols}")
print(f"All date columns found: {date_cols}")


def classify_date_format(val: str) -> str:
    s = val.strip()
    if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{2}", s):
        return "M/D/YY"
    if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", s):
        return "M/D/YYYY"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return "YYYY-MM-DD"
    if re.fullmatch(r"\d{1,2}-\d{1,2}-\d{2,4}", s):
        return "M-D-YY(YY)"
    return f"OTHER({s})"


parsed_dates = {}
for c in date_cols:
    raw = meta_df[c].dropna().astype(str).str.strip()
    fmt_counts = raw.map(classify_date_format).value_counts().to_dict()
    parsed = pd.to_datetime(meta_df[c], format="mixed", errors="coerce")
    parsed_dates[c] = parsed
    n_non_null = int(meta_df[c].notna().sum())
    n_parsed = int(parsed.notna().sum())
    n_failed = n_non_null - n_parsed
    year_mismatch = int((parsed.dt.year.notna() & (parsed.dt.year != meta_df["Year"])).sum())
    print(f"\n--- {c} ---")
    print(f"  Non-null: {n_non_null} | Parsed valid: {n_parsed} | Unparseable: {n_failed} | Missing: {int(meta_df[c].isna().sum())}")
    print(f"  Format distribution: {fmt_counts}")
    print(f"  Parsed year != Trial Year rows: {year_mismatch}")
    if year_mismatch > 0:
        mm = meta_df.loc[parsed.dt.year.notna() & (parsed.dt.year != meta_df["Year"]), ["Year", "Env", c]].copy()
        mm["parsed_date"] = parsed.loc[mm.index].dt.strftime("%Y-%m-%d")
        print(mm.to_string(index=False))

# Diagnostic station-deployment duration (Date_weather_station_removed - Date_weather_station_placed)
placed_dt = parsed_dates["Date_weather_station_placed"]
removed_dt = parsed_dates["Date_weather_station_removed"]
duration_days = (removed_dt - placed_dt).dt.days
valid_dur = duration_days.dropna()

print("\n=== WEATHER STATION DEPLOYMENT DURATION (DIAGNOSTIC ONLY — NOT PERSISTED) ===")
print(f"Pairs with both valid dates   : {len(valid_dur):,}")
print(f"Negative durations (< 0 days) : {int((valid_dur < 0).sum())}")
print(f"Zero durations (== 0 days)    : {int((valid_dur == 0).sum())}")
print(f"Durations > 365 days          : {int((valid_dur > 365).sum())}")
print(valid_dur.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2).to_string())

if (valid_dur <= 0).any() or (valid_dur > 365).any():
    weird_dur = meta_df.loc[duration_days.notna() & ((duration_days <= 0) | (duration_days > 365)),
                            ["Year", "Env", "Date_weather_station_placed", "Date_weather_station_removed"]].copy()
    weird_dur["duration_days"] = duration_days.loc[weird_dur.index].astype(int)
    print("\nUnusual station deployment durations (<= 0 or > 365 days):")
    print(weird_dur.to_string(index=False))

In [ ]:
# Audit available agricultural & management-related columns
mgmt_cols = [
    "Treatment",
    "Irrigated",
    "Previous_Crop",
    "Pre-plant_tillage_method(s)",
    "In-season_tillage_method(s)",
    "Type_of_planter (fluted cone; belt cone; air planter)",
    "Cardinal_Heading_Pass_1",
    "Soil_Taxonomic_ID and horizon description, if known",
    "System_Determining_Moisture",
    "Pounds_Needed_Grain_Moisture",
]

print("=== MANAGEMENT & FIELD-OPERATION VARIABLES SUMMARY ===")
for col in mgmt_cols:
    s = meta_df[col]
    n_non_null = int(s.notna().sum())
    miss_pct = float(s.isna().mean() * 100)
    n_raw = int(s.nunique(dropna=True))
    n_norm = int(s.dropna().astype(str).str.strip().str.lower().nunique())
    top_vals = s.dropna().astype(str).str.strip().value_counts().head(6).to_dict()
    print(f"\n{col}")
    print(f"  Non-null: {n_non_null} ({100 - miss_pct:.1f}%) | Missing: {int(s.isna().sum())} ({miss_pct:.1f}%) | Unique raw: {n_raw} | Unique normalized: {n_norm}")
    print(f"  Top values: {top_vals}")

issue_cols = [c for c in meta_df.columns if c.startswith("Issue/comment")]
any_issue = meta_df[issue_cols].notna().any(axis=1)
print(f"\n=== ISSUE/COMMENT COLUMNS ({len(issue_cols)} columns) ===")
print(f"Environments with >= 1 Issue/comment entry: {int(any_issue.sum())} / {len(meta_df)} ({any_issue.mean() * 100:.2f}%)")
print(meta_df[issue_cols].notna().sum().to_string())

In [ ]:
# Local visualization: Weather station placement/removal Day-of-Year & deployment duration
placed_doy = placed_dt.dropna().dt.day_of_year
removed_doy = removed_dt.dropna().dt.day_of_year

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(placed_doy, bins=25, alpha=0.75, color="#2f855a", label=f"Placed (n={len(placed_doy)})")
axes[0].hist(removed_doy, bins=25, alpha=0.75, color="#c53030", label=f"Removed (n={len(removed_doy)})")
axes[0].set_title("Weather Station Placement vs. Removal Day-of-Year")
axes[0].set_xlabel("Day of Year (1–366)")
axes[0].set_ylabel("Environment Count")
axes[0].legend(frameon=True)

axes[1].hist(valid_dur, bins=25, color="#3182ce", alpha=0.85, edgecolor="white")
axes[1].axvline(valid_dur.median(), color="#c53030", linestyle="--", linewidth=1.5, label=f"Median = {valid_dur.median():.0f} days")
axes[1].set_title(f"Weather Station Deployment Duration (n = {len(valid_dur)})")
axes[1].set_xlabel("Days (Removed - Placed)")
axes[1].set_ylabel("Environment Count")
axes[1].legend(frameon=True)

plt.tight_layout()
fig_path_4 = LOCAL_FIGURES_DIR / "04_meta_station_dates.png"
fig.savefig(fig_path_4, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved local figure: {fig_path_4.relative_to(PROJECT_ROOT).as_posix()}")

## 6. Cross-Dataset Identifier Coverage Audit (Trait vs. Metadata)

### Why one-to-many / many-to-one key audits matter for machine learning
In relational ML pipelines, joining plot-level trait records (`many` rows per `Env`) to environment-level metadata (`one` row per `Env`) is a **many-to-one** relationship:
- If metadata had duplicate `Env` keys, a join would silently duplicate training plots and inflate sample counts and CV scores.
- If a trait `Env` is missing from metadata, an **inner join** would silently drop valid yield labels, whereas a **left join** preserves all trait rows and introduces `NaN` metadata values for the unmatched environment(s).

Here we perform a strictly **read-only** audit using only `Env`, `Year`, `Field_Location`, and `Yield_Mg_ha` from `1_Training_Trait_Data_2014_2025.csv`, without creating or saving any merged table.

In [ ]:
# Read-only load of strictly required identifier + target columns from the trait table
trait_keys = pd.read_csv(
    TRAIT_PATH,
    engine="c",
    usecols=["Env", "Year", "Field_Location", "Yield_Mg_ha"],
    low_memory=False,
)
trait_finite_yield = np.isfinite(pd.to_numeric(trait_keys["Yield_Mg_ha"], errors="coerce"))

meta_envs = set(meta_df["Env"].dropna())
trait_envs = set(trait_keys["Env"].dropna())
matched_envs = meta_envs & trait_envs
trait_missing_from_meta = sorted(trait_envs - meta_envs)
meta_missing_from_trait = sorted(meta_envs - trait_envs)

affected_trait_mask = ~trait_keys["Env"].isin(meta_envs)
n_affected_trait_obs = int(affected_trait_mask.sum())
n_affected_valid_yield = int((affected_trait_mask & trait_finite_yield).sum())

# Verify many-to-one cardinality
obs_per_matched_env = trait_keys.loc[~affected_trait_mask].groupby("Env").size()
is_many_to_one = bool(meta_df["Env"].is_unique and obs_per_matched_env.max() > 1)

print("=== CROSS-DATASET Env COVERAGE AUDIT (READ-ONLY) ===")
print(f"Unique Env in Trait dataset              : {len(trait_envs):,}")
print(f"Unique Env in Metadata dataset           : {len(meta_envs):,}")
print(f"Matching Env identifiers                 : {len(matched_envs):,}")
print(f"Trait Envs missing from Metadata         : {len(trait_missing_from_meta)} -> {trait_missing_from_meta}")
print(f"Metadata Envs missing from Trait         : {len(meta_missing_from_trait)} -> {meta_missing_from_trait}")
print(f"Trait observations affected (no metadata): {n_affected_trait_obs:,} / {len(trait_keys):,} ({n_affected_trait_obs / len(trait_keys) * 100:.4f}%)")
print(f"Valid finite yield obs affected          : {n_affected_valid_yield:,} / {int(trait_finite_yield.sum()):,} ({n_affected_valid_yield / int(trait_finite_yield.sum()) * 100:.4f}%)")
print(f"Metadata Env key is strictly unique      : {meta_df['Env'].is_unique}")
print(f"Trait rows per matched Env (min/med/max) : {int(obs_per_matched_env.min())} / {obs_per_matched_env.median():.1f} / {int(obs_per_matched_env.max())}")
print(f"Confirmed many-to-one (Trait -> Meta)    : {is_many_to_one}")

# Compare Trait Field_Location vs Metadata Experiment_Code on matched Envs (without creating a persisted merged table)
trait_env_loc = trait_keys.groupby("Env")["Field_Location"].nunique()
print(f"\nTrait Envs with >1 Field_Location value  : {int((trait_env_loc > 1).sum())}")
trait_loc_map = trait_keys.drop_duplicates("Env").set_index("Env")["Field_Location"]
meta_exp_map = meta_df.set_index("Env")["Experiment_Code"]
common_idx = sorted(matched_envs)
loc_vs_exp_diff = pd.DataFrame({
    "Env": common_idx,
    "Trait_Field_Location": trait_loc_map.loc[common_idx].values,
    "Meta_Experiment_Code": meta_exp_map.loc[common_idx].values,
})
loc_vs_exp_mismatch = loc_vs_exp_diff[loc_vs_exp_diff["Trait_Field_Location"] != loc_vs_exp_diff["Meta_Experiment_Code"]]
print(f"Matched Envs where Trait Field_Location != Meta Experiment_Code: {len(loc_vs_exp_mismatch)}")
if len(loc_vs_exp_mismatch) > 0:
    print(loc_vs_exp_mismatch.to_string(index=False))

## 7. Preliminary Feature Eligibility & Leakage Assessment

### Guardrail
A metadata column being populated in historical 2014–2025 training data does **NOT** make it eligible for modeling. Final eligibility depends strictly on:
1. Verification against the **2026 testing metadata schema** (whether the field is actually provided for 2026 test environments with consistent definitions/completeness).
2. Confirmation that the column does not encode **in-season or post-harvest outcomes** (such as harvest moisture sensors, station removal dates, or post-hoc issue/comment logs describing drought, hail, or crop failure).

In [ ]:
classification = [
    # Identifiers
    ("Env", "Identifiers", "Primary environment key across G2F tables; join key and CV grouping key."),
    ("Experiment_Code", "Identifiers", "Site/trial-series identifier; audit mismatches against Env prefix and spacing variants."),
    ("Trial_ID (Assigned by collaborator for internal reference)", "Identifiers", "Internal collaborator trial ID; non-generalizable identifier."),
    ("Weather_Station_Serial_Number (Last four digits, e.g. m2700s#####)", "Identifiers", "Hardware serial number of weather station; non-causal equipment identifier."),
    # Potential temporal features
    ("Year", "Potential temporal features", "Trial calendar year; essential for temporal CV splits; not a stationary predictor alone."),
    ("Date_weather_station_placed", "Potential temporal features", "Weather-station deployment date; verify parsing consistency and availability in 2026 metadata."),
    # Potential geographical features
    ("Weather_Station_Latitude (in decimal numbers NOT DMS)", "Potential geographical features", "Decimal latitude of trial weather station; verify coverage in 2026 test metadata."),
    ("Weather_Station_Longitude (in decimal numbers NOT DMS)", "Potential geographical features", "Decimal longitude of trial weather station; verify coverage in 2026 test metadata."),
    ("Latitude_of_Field_Corner_#1 (lower left)", "Potential geographical features", "Field bounding-box corner latitude; verify coverage in 2026 test metadata."),
    ("Longitude_of_Field_Corner_#1 (lower left)", "Potential geographical features", "Field bounding-box corner longitude; verify coverage in 2026 test metadata."),
    ("Latitude_of_Field_Corner_#2 (lower right)", "Potential geographical features", "Field bounding-box corner latitude; verify coverage in 2026 test metadata."),
    ("Longitude_of_Field_Corner_#2 (lower right)", "Potential geographical features", "Field bounding-box corner longitude; verify coverage in 2026 test metadata."),
    ("Latitude_of_Field_Corner_#3 (upper right)", "Potential geographical features", "Field bounding-box corner latitude; verify coverage in 2026 test metadata."),
    ("Longitude_of_Field_Corner_#3 (upper right)", "Potential geographical features", "Field bounding-box corner longitude; verify coverage in 2026 test metadata."),
    ("Latitude_of_Field_Corner_#4 (upper left)", "Potential geographical features", "Field bounding-box corner latitude; verify coverage in 2026 test metadata."),
    ("Longitude_of_Field_Corner_#4 (upper left)", "Potential geographical features", "Field bounding-box corner longitude; verify coverage in 2026 test metadata."),
    ("City", "Potential geographical features", "Trial city name; useful for geocoding/sanity checks; verify normalization & 2026 overlap."),
    ("Farm", "Potential geographical features", "Research farm/station name; high-cardinality site descriptor."),
    ("Field", "Potential geographical features", "Specific field/parcel label; mostly site-year specific."),
    # Potential management features
    ("Treatment", "Potential management features", "Agronomic trial treatment; normalize casing/naming and verify availability in 2026 test metadata."),
    ("Irrigated", "Potential management features", "Irrigation indicator; potentially informative if reliably provided for 2026 test environments."),
    ("Previous_Crop", "Potential management features", "Prior-season crop rotation; normalize casing/spelling and verify 2026 availability."),
    ("Pre-plant_tillage_method(s)", "Potential management features", "Pre-plant tillage free text; high cardinality; verify 2026 availability."),
    ("In-season_tillage_method(s)", "Potential management features", "In-season cultivation text; verify whether planned pre-prediction or recorded in-season."),
    ("Type_of_planter (fluted cone; belt cone; air planter)", "Potential management features", "Planter equipment type; verify 2026 availability."),
    ("Cardinal_Heading_Pass_1", "Potential management features", "Field row/pass compass heading; heterogeneous formats; verify 2026 availability."),
    # Potential outcome-derived or unavailable-at-prediction-time features (PROVISIONALLY EXCLUDED)
    ("Date_weather_station_removed", "Potential outcome-derived or unavailable-at-prediction-time features", "Post-season/post-harvest station removal date; provisionally excluded pending 2026 test-schema check."),
    ("System_Determining_Moisture", "Potential outcome-derived or unavailable-at-prediction-time features", "Harvest moisture-sensor system; harvest-stage instrumentation, provisionally excluded."),
    ("Pounds_Needed_Grain_Moisture", "Potential outcome-derived or unavailable-at-prediction-time features", "Harvest moisture-sensor threshold; harvest-stage instrumentation, provisionally excluded."),
    ("Issue/comment_#1", "Potential outcome-derived or unavailable-at-prediction-time features", "Free-text field log; frequently records in-season/post-harvest stress or failures — provisionally excluded."),
    ("Issue/comment_#2", "Potential outcome-derived or unavailable-at-prediction-time features", "Free-text field log; high leakage/availability risk — provisionally excluded."),
    ("Issue/comment_#3", "Potential outcome-derived or unavailable-at-prediction-time features", "Free-text field log; high leakage/availability risk — provisionally excluded."),
    ("Issue/comment_#4", "Potential outcome-derived or unavailable-at-prediction-time features", "Free-text field log; high leakage/availability risk — provisionally excluded."),
    ("Issue/comment_#5", "Potential outcome-derived or unavailable-at-prediction-time features", "Free-text field log; high leakage/availability risk — provisionally excluded."),
    ("Issue/comment_#6", "Potential outcome-derived or unavailable-at-prediction-time features", "Free-text field log; high leakage/availability risk — provisionally excluded."),
    ("Issue/comment_#7", "Potential outcome-derived or unavailable-at-prediction-time features", "Free-text field log; high leakage/availability risk — provisionally excluded."),
    ("Issue/comment_#8", "Potential outcome-derived or unavailable-at-prediction-time features", "Free-text field log; high leakage/availability risk — provisionally excluded."),
    # Unknown fields requiring further investigation
    ("Soil_Taxonomic_ID and horizon description, if known", "Unknown fields requiring further investigation", "Free-text soil taxonomy/horizon notes; check overlap with soil dataset and 2026 test metadata."),
]

elig_df = pd.DataFrame(classification, columns=["column", "category", "rationale"])
elig_df["missing_pct"] = [round(float(meta_df[c].isna().mean() * 100), 2) for c in elig_df["column"]]
assert set(elig_df["column"]) == set(meta_df.columns), "Every metadata column must be classified!"
print("=== PRELIMINARY METADATA ELIGIBILITY & LEAKAGE CLASSIFICATION ===")
for cat, grp in elig_df.groupby("category", sort=False):
    print(f"  {cat} ({len(grp)}): {list(grp['column'])}")

In [ ]:
# Dynamic end-of-notebook verification summary (zero hardcoded statistics)
print("=== DYNAMIC PHASE 2.3 SUMMARY (COMPUTED LIVE) ===")
print(f"Metadata shape                                : {meta_df.shape[0]:,} rows x {meta_df.shape[1]} columns")
print(f"Load time / Deep memory                       : {load_seconds:.4f} s / {mem_mb_deep:.4f} MB")
print(f"Exact duplicate rows / Env unique             : {exact_dups:,} / {env_is_unique}")
print(f"Years covered                                 : {int(meta_df['Year'].min())}-{int(meta_df['Year'].max())} ({meta_df['Year'].nunique()} years)")
print(f"Unique Experiment_Code / City (raw)           : {meta_df['Experiment_Code'].nunique():,} / {meta_df['City'].nunique(dropna=True):,}")
print(f"Env prefix != Experiment_Code mismatches      : {len(prefix_vs_exp_mismatch):,}")
print(f"Envs with WS GPS / Any GPS / No GPS           : {int(has_ws.sum()):,} / {int(has_any_geo.sum()):,} / {int((~has_any_geo).sum()):,}")
print(f"Positive-longitude (Eastern Hem.) Envs        : {int(pos_lon_mask.sum()):,}")
print(f"Planting/Harvest/Plot-Area columns in meta    : {len(planting_cols)} / {len(harvest_cols)} / {len(plot_area_cols)}")
print(f"Station-date pairs with valid duration        : {len(valid_dur):,} (median {valid_dur.median():.1f} days)")
print(f"Trait Envs vs Meta Envs (Matched)             : {len(trait_envs):,} vs {len(meta_envs):,} ({len(matched_envs):,} matched)")
print(f"Trait Envs missing from Meta                  : {len(trait_missing_from_meta):,} -> {trait_missing_from_meta}")
print(f"Meta Envs missing from Trait                  : {len(meta_missing_from_trait):,}")
print(f"Trait obs / Valid yield obs without metadata  : {n_affected_trait_obs:,} / {n_affected_valid_yield:,}")
print(f"Many-to-one (Trait -> Meta) verified          : {is_many_to_one}")
print(f"Provisionally excluded meta columns           : {int((elig_df['category'] == 'Potential outcome-derived or unavailable-at-prediction-time features').sum())}")

## 8. Confirmed Qualitative Takeaways & Unresolved Technical Questions

*(All exact numerical findings are computed dynamically above and stored locally in `docs/METADATA_EDA.md` pending clarification of competition data-sharing permissions.)*

### Confirmed Structural Findings
- **Granularity & Relational Integrity:** `Env` uniquely identifies every metadata record with zero exact duplicate rows, confirming a **many-to-one** relationship from plot-level trait observations to environment-level metadata.
- **Imperfect Key Coverage:** A small mismatch exists between trait environments and metadata environments, confirming that future integration pipelines must use **left joins** (never silent inner joins) with explicit missing-metadata handling.
- **Schema Boundaries:** Planting dates, harvest dates, and plot areas do **not** appear in metadata; metadata instead tracks weather-station placement/removal dates, GPS coordinates (weather station + four field corners), site/location fields, agronomic management attributes, harvest-moisture instrumentation, and collaborator issue logs.
- **Text & Naming Heterogeneity:** Several categorical/identifier columns (`Experiment_Code`, `Treatment`, `Previous_Crop`, `City`, `Pre-plant_tillage_method(s)`, `Cardinal_Heading_Pass_1`) exhibit casing, spacing, or free-text variations that will require disciplined normalization only in Phase 3+.
- **Provisional Leakage Safeguards:** Harvest-stage instrumentation, post-season station removal dates, and free-text collaborator issue/comment columns are provisionally excluded from direct prediction inputs pending 2026 test-schema verification.

### Unresolved Technical Questions (For Phase 2.4+ / Phase 3)
1. **Missing Metadata Environment Handling:** Determine whether the unmatched trait environment appears in weather, soil, or EC tables and decide how to handle missing metadata during Phase 3 integration.
2. **2026 Test Metadata Schema Verification:** Confirm which metadata columns (coordinates, `Treatment`, `Irrigated`, `Previous_Crop`, tillage, planter, station placement dates) are actually populated and consistently formatted in 2026 test inputs.
3. **Coordinate Anomaly & Drift Verification:** Investigate environments with large weather-station-to-field-centroid distances, cross-year location coordinate spread, or missing GPS before deciding on spatial feature construction.
4. **Date Entry Anomalies:** Confirm whether station dates with year mismatches or `"All year"` tokens reflect multi-year permanent stations or collaborator typos.
5. **Data-Sharing Permissions:** Keep data-derived statistics, raw-record previews, and visualizations local until explicit competition data-sharing / redistribution permissions are clarified.